# 07 – Final Evaluation
## Held-out SemEval Gold Test Evaluation

Ebben a notebookban a már kiválasztott végleges modelleket értékeljük a korábban érintetlen SemEval-2016 gold test seten.

### Fagyasztott döntések

**Aspect Detection**
- class-weighted DistilBERT
- 12 aspektuskategória
- végleges validation threshold: `0.55`

**Aspect-Level Sentiment**
- application-compatible class-weighted DistilBERT
- input: `Sentence: ... Aspect: ...`
- output: `negative / neutral / positive`

### Fontos
A test eredményeket nem használjuk további tuningra vagy modellválasztásra.

## 1. Környezet

In [ ]:
!pip -q install "transformers>=4.46,<5" scikit-learn pandas numpy matplotlib

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

## 2. Input fájlok feltöltése

Töltsd fel:
- `EN_REST_SB1_TEST.xml.gold`
- `aspect_distilbert_weighted.zip`
- `sentiment_distilbert_application.zip`

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Nem Google Colab környezet.")

In [ ]:
TEST_PATH = Path("EN_REST_SB1_TEST.xml.gold")
ASPECT_ZIP = Path("aspect_distilbert_weighted.zip")
SENTIMENT_ZIP = Path("sentiment_distilbert_application.zip")
for path in [TEST_PATH, ASPECT_ZIP, SENTIMENT_ZIP]:
    assert path.exists(), f"Hiányzó fájl: {path}"
print("Mindhárom input fájl elérhető.")

## 3. Modellek kicsomagolása és betöltése

In [ ]:
ASPECT_MODEL_DIR = Path("aspect_model_final_eval")
SENTIMENT_MODEL_DIR = Path("sentiment_model_final_eval")
for folder in [ASPECT_MODEL_DIR, SENTIMENT_MODEL_DIR]:
    if folder.exists(): shutil.rmtree(folder)
    folder.mkdir()
shutil.unpack_archive(ASPECT_ZIP, ASPECT_MODEL_DIR)
shutil.unpack_archive(SENTIMENT_ZIP, SENTIMENT_MODEL_DIR)
aspect_tokenizer = AutoTokenizer.from_pretrained(ASPECT_MODEL_DIR)
aspect_model = AutoModelForSequenceClassification.from_pretrained(ASPECT_MODEL_DIR).to(DEVICE)
aspect_model.eval()
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_DIR)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_DIR).to(DEVICE)
sentiment_model.eval()
print("Mindkét modell betöltve.")

## 4. Label mapping ellenőrzése

In [ ]:
ASPECTS = [aspect_model.config.id2label[i] for i in range(aspect_model.config.num_labels)]
SENTIMENT_LABELS = [sentiment_model.config.id2label[i] for i in range(sentiment_model.config.num_labels)]
print("Aspect labels:")
for aspect in ASPECTS: print("-", aspect)
print("\nSentiment labels:", SENTIMENT_LABELS)
assert len(ASPECTS) == 12
assert set(SENTIMENT_LABELS) == {"negative", "neutral", "positive"}

## 5. Gold test XML beolvasása

In [ ]:
def parse_semeval_xml(path):
    root = ET.parse(path).getroot()
    sentences, opinions = [], []
    for review in root.findall("Review"):
        review_id = review.attrib["rid"]
        sentences_node = review.find("sentences")
        if sentences_node is None: continue
        for sentence in sentences_node.findall("sentence"):
            sentence_id = sentence.attrib["id"]
            text = sentence.findtext("text", default="")
            out_of_scope = sentence.attrib.get("OutOfScope", "FALSE").upper() == "TRUE"
            sentences.append({"review_id": review_id, "sentence_id": sentence_id, "text": text, "out_of_scope": out_of_scope})
            opinions_node = sentence.find("Opinions")
            if opinions_node is None: continue
            for opinion in opinions_node.findall("Opinion"):
                opinions.append({"review_id": review_id, "sentence_id": sentence_id, "text": text, "category": opinion.attrib.get("category"), "polarity": opinion.attrib.get("polarity"), "out_of_scope": out_of_scope})
    return pd.DataFrame(sentences), pd.DataFrame(opinions)

test_sentences, test_opinions = parse_semeval_xml(TEST_PATH)
print("Test mondatok:", len(test_sentences))
print("OutOfScope mondatok:", int(test_sentences["out_of_scope"].sum()))
print("Opinion annotációk:", len(test_opinions))

### Várt kontrollszámok
- Test mondatok: `676`
- OutOfScope mondatok: `34`
- Opinion annotációk: `859`

# 6. Final Aspect Detection evaluation

In [ ]:
def build_aspect_df(sentences_df, opinions_df, aspects):
    df = sentences_df.loc[~sentences_df["out_of_scope"]].copy()
    sentence_aspects = opinions_df.loc[~opinions_df["out_of_scope"]].groupby("sentence_id")["category"].apply(set).to_dict()
    for aspect in aspects:
        df[aspect] = df["sentence_id"].map(lambda sid: int(aspect in sentence_aspects.get(sid, set())))
    return df.reset_index(drop=True)

test_aspect_df = build_aspect_df(test_sentences, test_opinions, ASPECTS)
print("Aspect test mondatok:", len(test_aspect_df))
print("Gold aspect annotációk:", int(test_aspect_df[ASPECTS].to_numpy().sum()))

In [ ]:
FINAL_ASPECT_THRESHOLD = 0.55
ASPECT_BATCH_SIZE = 64
texts = test_aspect_df["text"].fillna("").tolist()
aspect_prob_batches = []
for start in range(0, len(texts), ASPECT_BATCH_SIZE):
    batch_texts = texts[start:start+ASPECT_BATCH_SIZE]
    inputs = aspect_tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.no_grad(): logits = aspect_model(**inputs).logits
    aspect_prob_batches.append(torch.sigmoid(logits).cpu().numpy())
aspect_test_probs = np.vstack(aspect_prob_batches)
aspect_test_preds = (aspect_test_probs >= FINAL_ASPECT_THRESHOLD).astype(int)
aspect_test_labels = test_aspect_df[ASPECTS].to_numpy(dtype=int)
print("Prediction matrix:", aspect_test_preds.shape)
print("Gold matrix:", aspect_test_labels.shape)

In [ ]:
aspect_metrics = {}
for average in ["micro", "macro"]:
    p,r,f1,_ = precision_recall_fscore_support(aspect_test_labels, aspect_test_preds, average=average, zero_division=0)
    aspect_metrics[f"precision_{average}"] = p
    aspect_metrics[f"recall_{average}"] = r
    aspect_metrics[f"f1_{average}"] = f1
aspect_metrics["subset_accuracy"] = accuracy_score(aspect_test_labels, aspect_test_preds)
aspect_metrics["hamming_loss"] = hamming_loss(aspect_test_labels, aspect_test_preds)
print("FINAL ASPECT TEST RESULTS")
for key,value in aspect_metrics.items(): print(f"{key}: {value:.4f}")

In [ ]:
p,r,f1,support = precision_recall_fscore_support(aspect_test_labels, aspect_test_preds, average=None, zero_division=0)
aspect_per_class = pd.DataFrame({"aspect":ASPECTS,"precision":p,"recall":r,"f1":f1,"support":support}).sort_values("support", ascending=False)
display(aspect_per_class)

# 7. Final Sentiment Classification evaluation

In [ ]:
def build_application_sentiment_df(opinions_df):
    df = opinions_df.loc[~opinions_df["out_of_scope"]].copy()
    df = df.drop_duplicates(subset=["review_id","sentence_id","text","category","polarity"]).copy()
    key_cols = ["review_id","sentence_id","category"]
    polarity_counts = df.groupby(key_cols)["polarity"].nunique().rename("n_polarities").reset_index()
    ambiguous_keys = polarity_counts.loc[polarity_counts["n_polarities"] > 1, key_cols].copy()
    if len(ambiguous_keys) > 0:
        marker = ambiguous_keys.assign(ambiguous=True)
        df = df.merge(marker, on=key_cols, how="left")
        df = df.loc[df["ambiguous"].isna()].drop(columns="ambiguous")
    df = df.drop_duplicates(subset=key_cols).reset_index(drop=True)
    return df, ambiguous_keys

test_sentiment_df, test_sentiment_ambiguous = build_application_sentiment_df(test_opinions)
print("Tiszta test sentence + aspect példák:", len(test_sentiment_df))
print("Kizárt ambivalens test kulcsok:", len(test_sentiment_ambiguous))

In [ ]:
sentiment_label2id = {label:i for i,label in enumerate(SENTIMENT_LABELS)}
test_sentiment_df["model_input"] = test_sentiment_df.apply(lambda row: f"Sentence: {row['text']} Aspect: {row['category']}.", axis=1)
test_sentiment_df["label"] = test_sentiment_df["polarity"].map(sentiment_label2id).astype(int)
display(test_sentiment_df[["model_input","polarity","label"]].head())

In [ ]:
SENTIMENT_BATCH_SIZE = 64
sentiment_pred_batches, sentiment_prob_batches = [], []
sentiment_inputs = test_sentiment_df["model_input"].fillna("").tolist()
for start in range(0, len(sentiment_inputs), SENTIMENT_BATCH_SIZE):
    batch_texts = sentiment_inputs[start:start+SENTIMENT_BATCH_SIZE]
    inputs = sentiment_tokenizer(batch_texts, padding=True, truncation=True, max_length=160, return_tensors="pt")
    inputs = {k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.no_grad(): logits = sentiment_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    sentiment_prob_batches.append(probs)
    sentiment_pred_batches.append(np.argmax(probs, axis=-1))
sentiment_test_probs = np.vstack(sentiment_prob_batches)
sentiment_test_preds = np.concatenate(sentiment_pred_batches)
sentiment_test_labels = test_sentiment_df["label"].to_numpy(dtype=int)
print("Sentiment test examples:", len(sentiment_test_labels))

In [ ]:
sentiment_accuracy = accuracy_score(sentiment_test_labels, sentiment_test_preds)
p_macro,r_macro,f1_macro,_ = precision_recall_fscore_support(sentiment_test_labels, sentiment_test_preds, labels=list(range(len(SENTIMENT_LABELS))), average="macro", zero_division=0)
_,_,f1_weighted,_ = precision_recall_fscore_support(sentiment_test_labels, sentiment_test_preds, labels=list(range(len(SENTIMENT_LABELS))), average="weighted", zero_division=0)
sentiment_metrics = {"accuracy":sentiment_accuracy,"precision_macro":p_macro,"recall_macro":r_macro,"f1_macro":f1_macro,"f1_weighted":f1_weighted}
print("FINAL SENTIMENT TEST RESULTS")
for key,value in sentiment_metrics.items(): print(f"{key}: {value:.4f}")

In [ ]:
p,r,f1,support = precision_recall_fscore_support(sentiment_test_labels, sentiment_test_preds, labels=list(range(len(SENTIMENT_LABELS))), average=None, zero_division=0)
sentiment_per_class = pd.DataFrame({"sentiment":SENTIMENT_LABELS,"precision":p,"recall":r,"f1":f1,"support":support})
display(sentiment_per_class)

In [ ]:
cm = confusion_matrix(sentiment_test_labels, sentiment_test_preds, labels=list(range(len(SENTIMENT_LABELS))))
sentiment_cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in SENTIMENT_LABELS], columns=[f"pred_{l}" for l in SENTIMENT_LABELS])
display(sentiment_cm_df)

# 8. Összefoglaló

In [ ]:
final_evaluation_summary = pd.DataFrame([
    {"component":"Aspect Detection","metric":"micro_precision","test_value":aspect_metrics["precision_micro"]},
    {"component":"Aspect Detection","metric":"micro_recall","test_value":aspect_metrics["recall_micro"]},
    {"component":"Aspect Detection","metric":"micro_f1","test_value":aspect_metrics["f1_micro"]},
    {"component":"Aspect Detection","metric":"macro_f1","test_value":aspect_metrics["f1_macro"]},
    {"component":"Sentiment Classification","metric":"accuracy","test_value":sentiment_metrics["accuracy"]},
    {"component":"Sentiment Classification","metric":"macro_f1","test_value":sentiment_metrics["f1_macro"]},
    {"component":"Sentiment Classification","metric":"weighted_f1","test_value":sentiment_metrics["f1_weighted"]},
])
display(final_evaluation_summary)

# 9. Export

In [ ]:
FINAL_SUMMARY_PATH = Path("final_evaluation_summary.csv")
ASPECT_CLASS_PATH = Path("final_aspect_per_class_metrics.csv")
SENTIMENT_CLASS_PATH = Path("final_sentiment_per_class_metrics.csv")
CONFUSION_PATH = Path("final_sentiment_confusion_matrix.csv")
final_evaluation_summary.to_csv(FINAL_SUMMARY_PATH, index=False)
aspect_per_class.to_csv(ASPECT_CLASS_PATH, index=False)
sentiment_per_class.to_csv(SENTIMENT_CLASS_PATH, index=False)
sentiment_cm_df.to_csv(CONFUSION_PATH)
print("Export kész.")